In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from statsmodels.tsa.seasonal import seasonal_decompose
import keras
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler

1.	Exploratory Data Analysis (EDA)

o	Visualize trends, seasonality, and anomalies in the milk production data.
o	Check for any missing values or outliers.
o	Normalize or scale the data for neural network models.


In [ ]:
pd.read_csv('/content/monthly_milk_production.csv').head(),pd.read_csv('/content/monthly_milk_production.csv').tail()

In [ ]:
df=pd.read_csv('/content/monthly_milk_production.csv',parse_dates=True, index_col='Date')

In [ ]:
df.head()

In [ ]:
df.isnull().sum()

In [ ]:
df.boxplot()

In [ ]:
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(df[['Production']])

In [ ]:
plt.plot(df)

In [ ]:
result= seasonal_decompose(df['Production'],model='additive')
result.trend.plot()

In [ ]:
result.seasonal.plot()

Dataset is seasonal.

In [ ]:
df=pd.read_csv('/content/monthly_milk_production.csv',parse_dates=True, index_col='Date')

In [ ]:
df.head()

In [ ]:
min_max= MinMaxScaler()

In [ ]:
data=df[['Production']]
scaler = MinMaxScaler()
data_scaled = scaler.fit_transform(data)
data_scaled

2.	Data Preparation for Deep Learning

o	Create input-output sequences (time windows) suitable for training RNNs/LSTMs/GRUs.
o	Split data into training, validation, and test sets.
o	Reshape data for model input dimensions


In [ ]:
window_size = 12
def create_dataset(data, window_size):
    X, y = [], []
    for i in range(len(data) - window_size):
        X.append(data[i:i+window_size])
        y.append(data[i+window_size])
    return np.array(X), np.array(y)

In [ ]:
train_size = int(len(data) * 0.8)
train, test = data[:train_size], data[train_size:]

In [ ]:
X_train = X_train.reshape((X_train.shape[0], X_train.shape[1], 1))

In [ ]:
monthly_avg = df.groupby(df.index.month).mean()

# plot
plt.figure(figsize=(10, 6))
plt.plot(monthly_avg.index, monthly_avg['Production'], marker='o')
plt.title('Monthly Average Milk Production')
plt.xlabel('Month')
plt.ylabel('Average Production')
plt.grid(True)
plt.show()

Here’s the Monthly Average milk production plot!

📊 You can observe the seasonality effect — for example:

*   Some months have higher production(likely summer or winter ).

*   Some months show lower usage.


This pattern is exactly why models like **RNNs** are important — they learn these time-based dependencies automatically!

3.	Model Building

o	Build three separate models:
	Basic RNN
	LSTM
	GRU
o	Tune hyperparameters (e.g., window size, number of units, batch size, epochs).
o	Use appropriate loss functions and optimizers


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense

tf.random.set_seed(42)
# Build RNN Model
model = Sequential()
model.add(SimpleRNN(50, activation='tanh', input_shape=(X_train.shape[1], 1))) # first hidden layer
model.add(Dense(1)) #

In [ ]:
model.compile(optimizer='adam', loss='mse')

In [ ]:
history = model.fit(X_train, y_train, epochs=30, batch_size=32, validation_split=0.2, verbose=1)

In [ ]:
#plot Loss
plt.figure(figsize=(10,5))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('RNN Training and Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss (MSE)')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
last_n_days = data_scaled[-n_steps:]
last_n_days = last_n_days.reshape((1, n_steps, 1))
Predicted = model.predict(last_n_days, verbose=0)
Predicted_ = scaler.inverse_transform(Predicted)
print(f"Predicted next month's milk production: {Predicted_[0][0]:.2f}")

LSTM Model

In [ ]:
tf.random.set_seed(42)
lstm=tf.keras.models.Sequential()
lstm.add(tf.keras.layers.LSTM(units=50,return_sequences=True,input_shape=(X_train.shape[1],1)))
lstm.add(tf.keras.layers.LSTM(units=50,return_sequences=True))
lstm.add(tf.keras.layers.Dense(units=1))

In [ ]:
lstm.summary()

In [ ]:
lstm.compile(optimizer=tf.keras.optimizers.Adam(),loss=tf.keras.losses.mse)

In [ ]:
history=lstm.fit(X_train,y_train,epochs=20,batch_size=40,validation_split=0.2)

In [ ]:

# Plot Loss
plt.figure(figsize=(10,5))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('LSTM Training and Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss (MSE)')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
last_n_days = data_scaled[-n_steps:]
last_n_days = last_n_days.reshape((1, n_steps, 1))
Predicted = lstm.predict(last_n_days, verbose=0)
# Extract the prediction for the last timestep and reshape to 2D
Predicted_ =  scaler.inverse_transform(Predicted[:, -1, :])
print(f"Predicted next month's milk production: {Predicted_[0][0]:.2f}")

GRU

In [ ]:
gru= tf.keras.models.Sequential()
gru.add(tf.keras.layers.GRU(units=50,return_sequences=True,input_shape=(X_train.shape[1],1)))
gru.add(tf.keras.layers.Dense(units=1))

In [ ]:
gru.compile(optimizer=tf.keras.optimizers.Adam(),loss=tf.keras.losses.mse)

In [ ]:
history=gru.fit(X_train,y_train,epochs=20,batch_size=40, validation_split=0.2)

In [ ]:

# Plot Loss
plt.figure(figsize=(10,5))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('GRU Training and Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss (MSE)')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
last_n_days = data_scaled[-n_steps:]
last_n_days = last_n_days.reshape((1, n_steps, 1))
Predicted = gru.predict(last_n_days, verbose=0)
# Extract the prediction for the last timestep and reshape to 2D
Predicted_ =  scaler.inverse_transform(Predicted[:, -1, :])
print(f"Predicted next month's milk production: {Predicted_[0][0]:.2f}")

4.	Model Evaluation:

o	Plot predictions vs. actual values.
o	Calculate forecasting metrics: RMSE, MAE, MAPE.

In [ ]:
rnn_pred = model.predict(X_test, verbose=0)
lstm_pred = lstm.predict(X_test, verbose=0)
gru_pred = gru.predict(X_test, verbose=0)

In [ ]:
rnn_pred_inv = scaler.inverse_transform(rnn_pred)
lstm_pred_inv = scaler.inverse_transform(lstm_pred[:, -1, :])
gru_pred_inv = scaler.inverse_transform(gru_pred[:, -1, :])
y_test_actual = scaler.inverse_transform(y_test.reshape(-1,1))

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np

def evaluate_model(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    return rmse, mae, mape

rnn_rmse, rnn_mae, rnn_mape = evaluate_model(y_test_actual, rnn_pred_inv)
lstm_rmse, lstm_mae, lstm_mape = evaluate_model(y_test_actual, lstm_pred_inv)
gru_rmse, gru_mae, gru_mape = evaluate_model(y_test_actual, gru_pred_inv)

In [ ]:
print("RNN  -> RMSE:", rnn_rmse, "MAE:", rnn_mae, "MAPE:", rnn_mape)
print("LSTM -> RMSE:", lstm_rmse, "MAE:", lstm_mae, "MAPE:", lstm_mape)
print("GRU  -> RMSE:", gru_rmse, "MAE:", gru_mae, "MAPE:", gru_mape)




RNN  -> RMSE: 15.711146381314725 MAE: 13.076779452237206 MAPE: 1.499760068418193

LSTM -> RMSE: 77.55088959273277 MAE: 63.30358054421165 MAPE: 7.083568942254305

GRU  -> RMSE: 63.79807632700391 MAE: 51.05924571644176 MAPE: 5.717468355218546


o	Compare the performance of RNN, LSTM, and GRU.
i) RNN:

Predicted next month's milk production: 818.47

ii)LSTM:

Predicted next month's milk production: 812.10

iii)GRU:

Predicted next month's milk production: 786.72










5.	Prediction and Visualization:

o	Forecast milk production for the next 12 months.

o	Visualize the predicted trend with uncertainty or confidence intervals if possible


In [ ]:
future_steps = 12
future_predictions = []

current_input = data_scaled[-n_steps:].reshape(1, n_steps, 1)

for _ in range(future_steps):
    pred = lstm.predict(current_input, verbose=0)
    # Append the last predicted value for the current step
    future_predictions.append(pred[0, -1, 0])

    # Shift the window: remove the oldest and add the newest prediction
    current_input = np.append(current_input[:, 1:, :], pred[:, -1:, :], axis=1)

future_predictions = scaler.inverse_transform(
    np.array(future_predictions).reshape(-1,1)
)

In [ ]:
plt.figure(figsize=(12,6))

plt.plot(df['Production'], label='Actual')
plt.plot(range(len(df), len(df)+12), future_predictions, label='Forecast')

plt.legend()
plt.title("Milk Production Forecast (Next 12 Months)")
plt.show()

6.	Business Insights

o	Interpret results and recommend how the dairy business can use these forecasts for better planning and resource allocation.



RNN:
Predicts the highest value (818.47)
Suggests slightly higher production trend
Based on earlier metrics → most accurate model

LSTM:
Prediction (812.10) is close to RNN
Indicates moderate production level
Slightly less accurate than RNN

GRU:
Predicts lowest value (786.72)
Suggests a more conservative estimate
Still reasonable but deviates more

Difference between models:

818.47 - 786.72 ≈ 31.75
Variation is small (~3–4%)
All models agree on overall trend direction

Based on earlier evaluation (RMSE, MAE, MAPE):

Best Model: RNN

The RNN model produced the most accurate forecasts based on evaluation metrics and predicts the next month’s milk production to be approximately 818.47 units. LSTM and GRU models also generated similar predictions, indicating consistency across models, but with slightly lower accuracy. Therefore, the RNN model is selected as the final model for forecasting.

Business Insight
Expected production ≈ 800–820 units
Helps in:
Inventory planning
Supply chain optimization
Demand forecasting
 Predictions are consistent
RNN is best-performing model
Final forecast ≈ 818 units
